# Multimodal Fashion Indexing Pipeline
This notebook implements the indexing pipeline using Qwen3-VL-2B-Instruct, Qwen2.5-0.5B, and BGE-M3, backed by PostgreSQL and FAISS.

In [2]:
from google.colab import drive
drive.mount('/content/drive')

# Replace this with the specific folder in your Google Drive where your dataset resides
DRIVE_PATH = '/content/drive/MyDrive/Fashion_search_engine'
import os
os.makedirs(DRIVE_PATH, exist_ok=True)
os.chdir(DRIVE_PATH)
print(f"Working directory changed to: {os.getcwd()}")


In [ ]:
# Core deps — quoted to avoid shell '>' redirection issue
!pip install -q "qwen-vl-utils>=0.0.8" "FlagEmbedding>=1.2.11" "faiss-cpu>=1.8.0" "psycopg2-binary>=2.9.9" "accelerate>=0.34.0" "tqdm>=4.66.0"

# Qwen3-VL requires transformers built from source (not yet on PyPI as of the model card)
!pip install -q git+https://github.com/huggingface/transformers
!pip show qwen-vl-utils

# PostgreSQL server setup
!sudo apt-get -y -qq update
!sudo apt-get -y -qq install postgresql postgresql-contrib
!sudo service postgresql start
!sudo -u postgres psql -c "CREATE USER fashion_user WITH SUPERUSER PASSWORD 'fashion_pass';"
!sudo -u postgres psql -c "CREATE DATABASE fashion_db OWNER fashion_user;"

In [3]:
import os
import json
import psycopg2
import torch
import faiss
import numpy as np
from PIL import Image
from tqdm import tqdm
from transformers import Qwen3VLForConditionalGeneration, AutoTokenizer, AutoProcessor, AutoModelForCausalLM
from FlagEmbedding import BGEM3FlagModel

print("transformers version check:")
import transformers
print(transformers.__version__)  

In [5]:
conn = psycopg2.connect(
    dbname="fashion_db",
    user="fashion_user",
    password="fashion_pass",
    host="127.0.0.1",
    port="5432"
)
cursor = conn.cursor()

cursor.execute('''
    CREATE TABLE IF NOT EXISTS images_metadata (
        faiss_id INTEGER PRIMARY KEY,
        image_id TEXT UNIQUE,
        image_path TEXT,
        original_caption TEXT,
        canonical_caption TEXT,
        metadata_json JSONB
    )
''')

# Enables fast structured pre-filtering (e.g. WHERE metadata_json->>'setting' = 'office')
cursor.execute('''
    CREATE INDEX IF NOT EXISTS idx_metadata_gin
    ON images_metadata USING GIN (metadata_json)
''')

conn.commit()
print("✅ Database and schema ready.")

In [5]:
# 1. VLM for image captioning
print("Loading Qwen3-VL-2B-Instruct...")
vlm_model_name = "Qwen/Qwen3-VL-2B-Instruct"
vlm_model = Qwen3VLForConditionalGeneration.from_pretrained(
    vlm_model_name,
    dtype="auto",
    device_map="auto",
)
vlm_processor = AutoProcessor.from_pretrained(vlm_model_name)

# 2. LLM for structured attribute extraction (canonicalization)
print("Loading Qwen2.5-0.5B-Instruct...")
llm_model_name = "Qwen/Qwen2.5-0.5B-Instruct"
llm_tokenizer = AutoTokenizer.from_pretrained(llm_model_name)
llm_model = AutoModelForCausalLM.from_pretrained(
    llm_model_name,
    dtype=torch.float16,
    device_map="auto"
)

# 3. Embedding model
print("Loading Embedding Model (BGE-M3)...")
embed_model = BGEM3FlagModel('BAAI/bge-m3', use_fp16=True)

# 4. FAISS index (HNSW for approximate search — scales better than flat index)
embedding_dim = 1024
USE_HNSW = True
if USE_HNSW:
    base_index = faiss.IndexHNSWFlat(embedding_dim, 32, faiss.METRIC_INNER_PRODUCT)
else:
    base_index = faiss.IndexFlatIP(embedding_dim)

print("✅ All models loaded.")

In [7]:
from qwen_vl_utils import process_vision_info
import torch
from PIL import Image

# Setup tokenizers for batch generation
llm_tokenizer.padding_side = 'left'
if llm_tokenizer.pad_token is None:
    llm_tokenizer.pad_token = llm_tokenizer.eos_token

if hasattr(vlm_processor, "tokenizer") and vlm_processor.tokenizer is not None:
    vlm_processor.tokenizer.padding_side = 'left'
    if vlm_processor.tokenizer.pad_token is None:
        vlm_processor.tokenizer.pad_token = vlm_processor.tokenizer.eos_token

# ============================================================
# Qwen3-VL : Batched Rich Caption Generation
# ============================================================
def generate_captions_batch(image_paths):
    prompt = """
You are a professional fashion image caption generator for an intelligent fashion search engine.

Describe the image in ONE clear, short, and accurate sentence.

Include ONLY the following if clearly visible:
- Upper body clothing with type and color (e.g., black shirt)
- Lower body clothing with type and color (e.g., blue jeans)
- Visible accessories with color (e.g., red tie, black hat)
- Background or environment if relevant (e.g., office, indoor, city street, park)
- Posture or action if visible (e.g., standing, walking, sitting)

Rules:
- Focus only on visible and factual details
- Do NOT guess, infer, or add extra information
- Do NOT describe emotions, style, or intent.
- Mention ONLY visible facts.
- Never guess hidden clothing.
- Never infer brands unless clearly visible.
- Never use adjectives like stylish, elegant, beautiful or fashionable.
- If something is uncertain, omit it.
- Start directly with the description.

Return ONLY one sentence.
"""

    messages_batch = []
    for path in image_paths:
        image = Image.open(path).convert("RGB")
        image.thumbnail((512, 512))
        
        messages_batch.append([
            {
                "role": "user",
                "content": [
                    {"type": "image", "image": image},
                    {"type": "text", "text": prompt}
                ]
            }
        ])

    texts = [
        vlm_processor.apply_chat_template(msg, tokenize=False, add_generation_prompt=True)
        for msg in messages_batch
    ]

    image_inputs, video_inputs = process_vision_info(messages_batch)

    inputs = vlm_processor(
        text=texts,
        images=image_inputs,
        videos=video_inputs,
        padding=True,
        return_tensors="pt"
    ).to(vlm_model.device)

    with torch.no_grad():
        generated_ids = vlm_model.generate(
            **inputs,
            max_new_tokens=128,
            do_sample=False
        )

    generated_ids_trimmed = [
        out_ids[len(in_ids):]
        for in_ids, out_ids in zip(inputs.input_ids, generated_ids)
    ]

    output_texts = vlm_processor.batch_decode(
        generated_ids_trimmed,
        skip_special_tokens=True,
        clean_up_tokenization_spaces=False
    )

    return [text.strip() for text in output_texts]


# ============================================================
# Qwen2.5 : Batched Semantic Text Normalization
# ============================================================
def normalize_texts_batch(raw_captions):
    system_prompt = """
You are a semantic text normalization model for an intelligent fashion retrieval system.

Extract ONLY essential fashion-related retrieval phrases.

Rules:
1. Extract clothing items only if explicitly mentioned.
2. Keep colors attached to their garments as ONE unit.
3. Keep garment attributes attached to garments.
4. Extract accessories only if explicitly visible.
5. Extract footwear only if explicitly visible.
6. Extract meaningful environments only if relevant. If a setting has a descriptive color/quality, output it as ONE merged phrase.
7. Ignore actions (walking, standing, sitting, posing, running).
8. Ignore body parts and hairstyles.
9. Ignore camera viewpoints.
10. Ignore background people.
11. Never infer style, occasion, weather or emotions.
12. Never invent garments, colors, accessories, or settings not present in the input.
13. Remove duplicate information.
14. If multiple colors belong to the same garment, keep them attached to that garment.
15. Preserve the exact descriptive words used in the input.
16. Output ONLY retrieval phrases separated by " | ".
17. Never output complete sentences.
"""

    base_messages = [
        {"role": "system", "content": system_prompt},
        {"role": "user", "content": "Input: A person wearing a bright yellow raincoat and black pants, standing outdoors on a city street.\nOutput:"},
        {"role": "assistant", "content": "yellow raincoat | black pants | city street"},
        {"role": "user", "content": "Input: A woman wearing a blue and white patterned jacket over a white ribbed top, blue plaid trousers and a brown beret against an orange studio background.\nOutput:"},
        {"role": "assistant", "content": "blue and white patterned jacket | white ribbed top | blue plaid trousers | brown beret | orange studio background"},
        {"role": "user", "content": "Input: A man wearing a navy blazer over a white dress shirt with a red tie inside a modern office.\nOutput:"},
        {"role": "assistant", "content": "navy blazer | white dress shirt | red tie | modern office"},
        {"role": "user", "content": "Input: A woman wearing a floral dress with a black jacket, black leggings and a colorful necklace while standing in front of a brown metal door.\nOutput:"},
        {"role": "assistant", "content": "floral dress | black jacket | black leggings | colorful necklace | brown metal door"},
        {"role": "user", "content": "Input: A man in a navy blue blazer, white dress shirt, red tie, tan pants and brown leather loafers walking down a city street.\nOutput:"},
        {"role": "assistant", "content": "navy blue blazer | white dress shirt | red tie | tan pants | brown leather loafers | city street"},
        {"role": "user", "content": "Input: A model walks on a runway wearing a colorful patterned jacket with green, white and black, paired with a black skirt and holding a black handbag.\nOutput:"},
        {"role": "assistant", "content": "green white black patterned jacket | black skirt | black handbag | runway"},
        {"role": "user", "content": "Input: A woman is walking on a runway wearing a sleeveless floral dress with a black lace hem, a gold necklace and her hair tied back.\nOutput:"},
        {"role": "assistant", "content": "floral dress | black lace hem | gold necklace | runway"},
        {"role": "user", "content": "Input: A woman wearing a red backless dress with intricate lace detailing and a sparkly full skirt, viewed from behind.\nOutput:"},
        {"role": "assistant", "content": "red backless dress | lace detail | sparkly full skirt"},
        {"role": "user", "content": "Input: A woman wearing a blue jacket with her hand on her hip while carrying a brown handbag.\nOutput:"},
        {"role": "assistant", "content": "blue jacket | brown handbag"},
        {"role": "user", "content": "Input: A woman stands in a studio with a warm yellow background, wearing a blue and white patterned jacket over a white ribbed top, blue plaid trousers, a brown beret, and a gold bracelet on her right wrist.\nOutput:"},
        {"role": "assistant", "content": "blue and white patterned jacket | white ribbed top | blue plaid trousers | brown beret | gold bracelet | warm yellow studio background"},
    ]

    batch_texts = []
    for raw_caption in raw_captions:
        msgs = list(base_messages)
        msgs.append({"role": "user", "content": f"Input: {raw_caption}\nOutput:"})
        text = llm_tokenizer.apply_chat_template(msgs, tokenize=False, add_generation_prompt=True)
        batch_texts.append(text)

    inputs = llm_tokenizer(batch_texts, return_tensors="pt", padding=True).to(llm_model.device)

    with torch.no_grad():
        outputs = llm_model.generate(**inputs, max_new_tokens=64, do_sample=False)

    generated_ids_trimmed = [
        out_ids[len(in_ids):]
        for in_ids, out_ids in zip(inputs.input_ids, outputs)
    ]

    normalized_texts = llm_tokenizer.batch_decode(generated_ids_trimmed, skip_special_tokens=True)

    cleaned_texts = []
    for norm in normalized_texts:
        cleaned = norm.strip().strip('"').strip("'")
        if cleaned.lower().startswith("output:"):
            cleaned = cleaned[7:].strip()
        cleaned_texts.append(cleaned)

    return cleaned_texts


In [ ]:
import os
import glob
import json
import shutil
import numpy as np
import pandas as pd
import faiss
from tqdm import tqdm
from PIL import Image
from IPython.display import display
from psycopg2.extras import execute_values

# ============================================================
# Configuration
# ============================================================

IMAGE_FOLDER = "./data/val_test2020/test"
image_paths = sorted(glob.glob(os.path.join(IMAGE_FOLDER, "*.jpg")))
print(f"Found {len(image_paths)} images.")

BACKUP_FOLDER = "/content/drive/MyDrive/FashionSearchEngine"
os.makedirs(BACKUP_FOLDER, exist_ok=True)

LOCAL_FAISS = "fashion_search_hnsw.faiss"
DRIVE_FAISS = os.path.join(BACKUP_FOLDER, "fashion_search_hnsw.faiss")
DRIVE_METADATA = os.path.join(BACKUP_FOLDER, "images_metadata.parquet")

SAVE_EVERY = 500  # Will trigger save when 'successful' hits multiples of this
DISPLAY_EVERY = 50
BATCH_SIZE = 8

# ============================================================
# Resume indexing
# ============================================================

cursor.execute("SELECT MAX(faiss_id) FROM images_metadata")
result = cursor.fetchone()[0]
current_faiss_id = result + 1 if result is not None else 0

successful = 0
failed = 0
skipped = 0

print(f"Starting FAISS ID : {current_faiss_id}")

def build_metadata_json(normalized_caption):
    phrases = [p.strip() for p in normalized_caption.split("|") if p.strip()]
    return {"phrases": phrases}

# ============================================================
# Main Batch Loop
# ============================================================

for i in tqdm(range(0, len(image_paths), BATCH_SIZE), desc="Indexing Batches"):
    batch_paths = image_paths[i:i + BATCH_SIZE]
    
    # ----------------------------
    # Filter already indexed images
    # ----------------------------
    batch_image_ids = [os.path.splitext(os.path.basename(p))[0] for p in batch_paths]
    
    cursor.execute("SELECT image_id FROM images_metadata WHERE image_id = ANY(%s)", (batch_image_ids,))
    existing = set(row[0] for row in cursor.fetchall())
    
    new_paths = [p for p in batch_paths if os.path.splitext(os.path.basename(p))[0] not in existing]
    skipped += (len(batch_paths) - len(new_paths))
    
    if not new_paths:
        continue
        
    try:
        # ----------------------------
        # Step 1 : Caption (Batch)
        # ----------------------------
        raw_captions = generate_captions_batch(new_paths)
        
        # ----------------------------
        # Step 2 : Normalize (Batch)
        # ----------------------------
        normalized_captions = normalize_texts_batch(raw_captions)
        
        # ----------------------------
        # Step 3 : BGE-M3 Embedding (Batch)
        # ----------------------------
        embedding_dict = embed_model.encode(normalized_captions, max_length=8192)
        embeddings = np.asarray(embedding_dict["dense_vecs"], dtype=np.float32)
        
        # ----------------------------
        # Step 4 : FAISS (Batch)
        # ----------------------------
        batch_size = len(new_paths)
        faiss_id_arr = np.arange(current_faiss_id, current_faiss_id + batch_size, dtype=np.int64)
        index.add_with_ids(embeddings, faiss_id_arr)
        
        # ----------------------------
        # Step 5 : PostgreSQL (Batch Insert)
        # ----------------------------
        records = []
        for idx in range(batch_size):
            image_id = os.path.splitext(os.path.basename(new_paths[idx]))[0]
            metadata_json = build_metadata_json(normalized_captions[idx])
            
            records.append((
                int(faiss_id_arr[idx]),
                image_id,
                new_paths[idx],
                raw_captions[idx],
                normalized_captions[idx],
                json.dumps(metadata_json)
            ))
            
        execute_values(
            cursor,
            """
            INSERT INTO images_metadata
            (faiss_id, image_id, image_path, original_caption, canonical_caption, metadata_json)
            VALUES %s
            """,
            records
        )
        conn.commit()
        
        current_faiss_id += batch_size
        successful += batch_size
        
        # ----------------------------
        # Visual Review
        # ----------------------------
        if successful % DISPLAY_EVERY < batch_size and successful >= DISPLAY_EVERY:
            img = Image.open(new_paths[-1])
            img.thumbnail((250,250))
            display(img)
            
            print("\nImage :", os.path.splitext(os.path.basename(new_paths[-1]))[0])
            print("\nRich Caption")
            print(raw_captions[-1])
            print("\nNormalized")
            print(normalized_captions[-1])
            print("="*80)
            
        # ----------------------------
        # Checkpoint
        # ----------------------------
        if successful % SAVE_EVERY < batch_size and successful >= SAVE_EVERY:
            print("\n💾 Saving checkpoint...")
            faiss.write_index(index, LOCAL_FAISS)
            shutil.copy(LOCAL_FAISS, DRIVE_FAISS)
            df = pd.read_sql_query("SELECT * FROM images_metadata", conn)
            df.to_parquet(DRIVE_METADATA, index=False)
            print("✅ Checkpoint Saved")

    except Exception as e:
        failed += len(new_paths)
        conn.rollback()
        print(f"\n❌ Error in batch starting at: {new_paths[0]}")
        print(e)

# ============================================================
# Final Save
# ============================================================

print("\nSaving Final Artifacts...")
faiss.write_index(index, LOCAL_FAISS)
shutil.copy(LOCAL_FAISS, DRIVE_FAISS)
df = pd.read_sql_query("SELECT * FROM images_metadata", conn)
df.to_parquet(DRIVE_METADATA, index=False)

print("\n===============================")
print("INDEXING FINISHED")
print("===============================")
print(f"Successful : {successful}")
print(f"Skipped    : {skipped}")
print(f"Failed     : {failed}")
print("\nSaved Files")
print(DRIVE_FAISS)
print(DRIVE_METADATA)


In [ ]:
import faiss
import pandas as pd

# Mount Drive in the new notebook (if not already mounted)
from google.colab import drive
drive.mount('/content/drive')

BACKUP_FOLDER = "/content/drive/MyDrive/FashionSearchEngine"

# Load FAISS index directly from Drive
index = faiss.read_index(f"{BACKUP_FOLDER}/fashion_search_hnsw.faiss")

# Load metadata directly from Drive — no Postgres needed
metadata_df = pd.read_parquet(f"{BACKUP_FOLDER}/images_metadata.parquet")

print(f"FAISS index loaded: {index.ntotal} vectors")
print(f"Metadata loaded: {len(metadata_df)} rows")

In [ ]:
# Save FAISS index
faiss.write_index(index, "fashion_index.faiss")
print("✅ FAISS index saved to fashion_index.faiss")

# Dump PostgreSQL data so it persists across Colab restarts
!pg_dump -h 127.0.0.1 -U fashion_user -d fashion_db -f fashion_db_dump.sql
print("✅ PostgreSQL database dumped to fashion_db_dump.sql")

In [ ]:
# If you want to dump the PostgreSQL data to a file so it persists when Colab restarts:
!pg_dump -h 127.0.0.1 -U fashion_user -d fashion_db -f fashion_db_dump.sql
print("PostgreSQL database dumped to fashion_db_dump.sql in your Drive.")